# 01_download.ipynb - 数据下载

本 Notebook 负责下载所有所需数据：
- 10只股票的日度行情数据（使用 baostock）
- 沪深300和中证全指的指数数据（使用 baostock）
- CPI和汇率的宏观数据（使用世界银行 API）
- 股票财务指标数据（使用模拟数据，世界银行无个股财务数据）

所有下载操作都会记录到 download_log.txt

In [11]:
import os
import sys
import time
import datetime
import baostock as bs
import pandas as pd
import numpy as np
import wbgapi as wb

# 项目路径设置
project_root = os.getcwd()
sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

Project root: c:\Users\86155\Desktop\新建文件夹\dshw-p01


In [12]:
# 配置信息
START_DATE = "2020-01-01"
END_DATE = datetime.datetime.now().strftime("%Y-%m-%d")

# 股票列表
STOCKS = [
    ("601398", "工商银行", "银行"),
    ("000001", "平安银行", "银行"),
    ("600104", "上汽集团", "汽车"),
    ("002594", "比亚迪", "汽车"),
    ("000002", "万科A", "房地产"),
    ("600048", "保利发展", "房地产"),
    ("600519", "贵州茅台", "白酒"),
    ("000858", "五粮液", "白酒"),
    ("601088", "中国神华", "能源"),
    ("600050", "中国联通", "通讯")
]

# 指数列表
INDICES = [
    ("000300", "沪深300"),
    ("000905", "中证500")
]

# 日志文件路径
LOG_FILE = os.path.join(project_root, "download_log.txt")

In [13]:
def log_message(status, item, message=""):
    """记录下载日志"""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] {status:8s} {item}"
    if message:
        log_line += f"  {message}"
    
    print(log_line)
    
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_line + "\n")

In [14]:
def convert_code(code):
    """将A股代码转换为baostock格式"""
    if code.startswith('6'):
        return f"sh.{code}"
    else:
        return f"sz.{code}"

## 1. 下载股票行情数据

In [15]:
def download_stock(code, name, start_date, end_date):
    """下载单只股票数据（使用baostock）"""
    try:
        # 登录baostock
        lg = bs.login()
        if lg.error_code != '0':
            return False, f"Login failed: {lg.error_msg}"
        
        # 转换代码格式
        bs_code = convert_code(code)
        
        # 获取后复权数据
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"  # 2=后复权
        )
        
        if rs.error_code != '0':
            bs.logout()
            return False, f"Query failed: {rs.error_msg}"
        
        # 转换为DataFrame
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        df = pd.DataFrame(data_list, columns=rs.fields)
        
        # 数据类型转换
        numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df = df[["date", "open", "close", "high", "low", "volume", "amount"]]
        
        # 保存为CSV
        output_path = os.path.join(project_root, "data", "stock", f"stock_{code}.csv")
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        
        bs.logout()
        return True, f"shape={df.shape}"
    except Exception as e:
        try:
            bs.logout()
        except:
            pass
        return False, f"Error: {str(e)}"

In [16]:
print("开始下载股票行情数据...")
for code, name, industry in STOCKS:
    time.sleep(1)
    success, message = download_stock(code, name, START_DATE, END_DATE)
    status = "SUCCESS" if success else "FAILED"
    log_message(status, f"stock_{code}", message)

开始下载股票行情数据...
login success!


KeyboardInterrupt: 

## 2. 下载指数数据

In [17]:
def download_index(code, name, start_date, end_date):
    """下载指数数据（使用baostock）"""
    try:
        # 登录baostock
        lg = bs.login()
        if lg.error_code != '0':
            return False, f"Login failed: {lg.error_msg}"
        
        # 指数代码格式
        bs_code = f"sh.{code}"
        
        # 获取指数数据
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="3"  # 3=不复权
        )
        
        if rs.error_code != '0':
            bs.logout()
            return False, f"Query failed: {rs.error_msg}"
        
        # 转换为DataFrame
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        df = pd.DataFrame(data_list, columns=rs.fields)
        
        # 数据类型转换
        numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df = df[["date", "open", "close", "high", "low", "volume", "amount"]]
        
        # 保存为CSV
        output_path = os.path.join(project_root, "data", "index", f"index_{code}.csv")
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        
        bs.logout()
        return True, f"shape={df.shape}"
    except Exception as e:
        try:
            bs.logout()
        except:
            pass
        return False, f"Error: {str(e)}"

In [18]:
print("开始下载指数数据...")
for code, name in INDICES:
    time.sleep(1)
    success, message = download_index(code, name, START_DATE, END_DATE)
    status = "SUCCESS" if success else "FAILED"
    log_message(status, f"index_{code}", message)

开始下载指数数据...
login success!
logout success!
[2026-04-08 19:29:16] SUCCESS  index_000300  shape=(1516, 7)
login success!
logout success!
[2026-04-08 19:29:24] SUCCESS  index_000905  shape=(1516, 7)


## 3. 下载宏观数据（使用世界银行 API）

In [ ]:
def download_cpi():
    """下载 CPI 同比增速数据（使用世界银行 API）"""
    try:
        # 世界银行指标：FP.CPI.TOTL.ZG - 中国通胀率（年度%）
        # 中国国家代码：CHN
        print("正在从世界银行获取 CPI 数据...")
        
        # 获取2020年至今的年度数据
        df_wb = wb.data.DataFrame(
            series=['FP.CPI.TOTL.ZG'],
            economy=['CHN'],
            time=range(2020, 2026),
            labels=True
        )
        
        if df_wb is None or len(df_wb) == 0:
            return False, "No data returned from World Bank"
        
        # 重置索引并整理数据
        df_wb = df_wb.reset_index()
        
        # 转换为月度数据（世界银行只有年度数据，我们将年度值分配到各月）
        monthly_data = []
        for year in range(2020, 2026):
            year_col = f"YR{year}"
            if year_col in df_wb.columns:
                cpi_value = df_wb[year_col].iloc[0]
                if pd.notna(cpi_value):
                    for month in range(1, 13):
                        monthly_data.append({
                            'date': f"{year}-{month:02d}",
                            'cpi_yoy': float(cpi_value)
                        })
        
        df = pd.DataFrame(monthly_data)
        
        # 只保留到当前月份
        current_month = datetime.datetime.now().strftime("%Y-%m")
        df = df[df['date'] <= current_month].copy()
        
        output_path = os.path.join(project_root, "data", "macro", "macro_cpi.csv")
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        
        return True, f"shape={df.shape}, source=World Bank (FP.CPI.TOTL.ZG)"
    except Exception as e:
        return False, f"Error: {str(e)}"

In [ ]:
def download_exchange_rate():
    """下载人民币/美元汇率数据（使用世界银行 API，替代M2）"""
    try:
        # 世界银行指标：PA.NUS.FCRF - 官方汇率（LCU per US$）
        print("正在从世界银行获取汇率数据...")
        
        # 获取2020年至今的年度数据
        df_wb = wb.data.DataFrame(
            series=['PA.NUS.FCRF'],
            economy=['CHN'],
            time=range(2020, 2026),
            labels=True
        )
        
        if df_wb is None or len(df_wb) == 0:
            return False, "No data returned from World Bank"
        
        # 重置索引并整理数据
        df_wb = df_wb.reset_index()
        
        # 转换为月度数据
        monthly_data = []
        for year in range(2020, 2026):
            year_col = f"YR{year}"
            if year_col in df_wb.columns:
                fx_value = df_wb[year_col].iloc[0]
                if pd.notna(fx_value):
                    for month in range(1, 13):
                        monthly_data.append({
                            'date': f"{year}-{month:02d}",
                            'fx_rate': float(fx_value)
                        })
        
        df = pd.DataFrame(monthly_data)
        
        # 只保留到当前月份
        current_month = datetime.datetime.now().strftime("%Y-%m")
        df = df[df['date'] <= current_month].copy()
        
        output_path = os.path.join(project_root, "data", "macro", "macro_fx.csv")
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        
        return True, f"shape={df.shape}, source=World Bank (PA.NUS.FCRF)"
    except Exception as e:
        return False, f"Error: {str(e)}"

In [ ]:
print("开始下载宏观数据...")

time.sleep(1)
success, message = download_cpi()
status = "SUCCESS" if success else "FAILED"
log_message(status, "macro_cpi", message)

time.sleep(1)
success, message = download_exchange_rate()
status = "SUCCESS" if success else "FAILED"
log_message(status, "macro_fx", message)

开始下载宏观数据...
正在从世界银行获取 CPI 数据...
[2026-04-08 15:01:24] SUCCESS  macro_cpi  shape=(60, 2), source=World Bank (FP.CPI.TOTL.ZG)
正在从世界银行获取汇率数据...
[2026-04-08 15:01:26] SUCCESS  macro_fx  shape=(60, 2), source=World Bank (PA.NUS.FCRF)


## 4. 下载财务数据

In [ ]:
def download_finance_data(stocks):
    """下载财务数据（直接使用baostock提供的ROE和净利润率）"""
    
    print("【1/2】正在下载财务数据...")
    finance_records = []
    
    # 登录baostock
    lg = bs.login()
    if lg.error_code != '0':
        return False, f"Login failed: {lg.error_msg}"
    
    total = len(stocks)
    for idx, (code, name, industry) in enumerate(stocks, 1):
        print(f"  [{idx}/{total}] 下载 {code} {name}...")
        try:
            bs_code = convert_code(code)
            
            for year in [2020, 2021, 2022, 2023, 2024]:
                # 获取利润表数据（baostock已经算好了ROE和净利润率）
                profit_rs = bs.query_profit_data(code=bs_code, year=year, quarter=4)
                
                if profit_rs.error_code == '0':
                    while (profit_rs.error_code == '0') & profit_rs.next():
                        row = profit_rs.get_row_data()
                        fields = profit_rs.fields
                        
                        # 转成字典方便访问
                        data_dict = dict(zip(fields, row))
                        
                        # 直接使用 baostock 提供的 ROE
                        if 'roeAvg' in data_dict and data_dict['roeAvg']:
                            roe = pd.to_numeric(data_dict['roeAvg'], errors='coerce')
                            if pd.notna(roe):
                                finance_records.append({
                                    'code': code,
                                    'year': str(year),
                                    'indicator': 'ROE',
                                    'value': roe * 100  # 转成百分比
                                })
                        
                        # 直接使用 baostock 提供的净利润率
                        if 'npMargin' in data_dict and data_dict['npMargin']:
                            np_margin = pd.to_numeric(data_dict['npMargin'], errors='coerce')
                            if pd.notna(np_margin):
                                finance_records.append({
                                    'code': code,
                                    'year': str(year),
                                    'indicator': 'net_profit_margin',
                                    'value': np_margin * 100  # 转成百分比
                                })
                        
        except Exception as e:
            print(f"    失败: {e}")
            continue
    
    bs.logout()
    print("【1/2】财务数据下载完成！")
    
    if finance_records:
        df_finance = pd.DataFrame(finance_records)
        # 补齐股票代码为6位
        df_finance['code'] = df_finance['code'].astype(str).str.zfill(6)
        output_path = os.path.join(project_root, "data", "finance", "finance_ratios.csv")
        df_finance.to_csv(output_path, index=False, encoding="utf-8-sig")
        return True, f"shape={df_finance.shape}"
    else:
        return False, "No finance data collected"


In [ ]:
print("开始下载财务数据...")
success, message = download_finance_data(STOCKS)
status = "SUCCESS" if success else "FAILED"
log_message(status, "finance_ratios", message)

开始下载财务数据...
【1/2】正在下载财务数据...
login success!
  [1/10] 下载 601398 工商银行...
  [2/10] 下载 000001 平安银行...
  [3/10] 下载 600104 上汽集团...
  [4/10] 下载 002594 比亚迪...
  [5/10] 下载 000002 万科A...
  [6/10] 下载 600048 保利发展...
  [7/10] 下载 600519 贵州茅台...
  [8/10] 下载 000858 五粮液...
  [9/10] 下载 601088 中国神华...
  [10/10] 下载 600050 中国联通...
logout success!
【1/2】财务数据下载完成！
[2026-04-08 19:19:39] SUCCESS  finance_ratios  shape=(100, 4)


## 下载完成

所有数据下载任务已完成，请查看 download_log.txt 确认下载状态。

数据来源说明：
- 股票行情：baostock，后复权，日度
- 市场指数：baostock
- 宏观指标：世界银行 API
  - CPI: World Bank (FP.CPI.TOTL.ZG) - Inflation, consumer prices (annual %)
  - 汇率: World Bank (PA.NUS.FCRF) - Official exchange rate (LCU per US$, period average)
  - 注：世界银行只有年度数据，已分配到各月
- 财务数据：净利润、净资产来自baostock，计算：ROE = 净利润 / 净资产，净利润率 = 净利润 / 营业收入